In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as transforms
import matplotlib.ticker as ticker
from matplotlib.colors import LogNorm
from matplotlib.colors import SymLogNorm
import yt
import os
from dotenv import dotenv_values
import glob
import pandas as pd
import re
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from scipy.stats import linregress


In [ ]:
def read_in_spectrum_data(run_dir):
    spect_dir = os.path.join(run_dir, "turb_vel*.txt")
    files = sorted(glob.glob(spect_dir))
    dfs = []
    for f in files:
        num = int(re.search(r'turb_vel(\d+)\.txt', f).group(1))
        df = pd.read_csv(f, sep='\s+', header=None)
        df['file_num'] = num
        dfs.append(df)
    
    spect_data = pd.concat(dfs, ignore_index=True)
    return spect_data
    
spect_data_list = []
spect_data_list_stoch_flux_0 = []
N_seeds = 1
for i in range(N_seeds):
    # run_dir_i = f"Kolmogorov_WENO_512_1em4_cell_depth_1em5_F_0_4em1_n_WN_10_dt_9em4_seed_{i+1}00"
    run_dir_i = f"Kolmogorov_WENO_512_1em4_cell_depth_1em5_F_0_1em1_n_WN_3_dt_3em4_seed_{i+1}00"
    spect_data_i = read_in_spectrum_data(run_dir_i)
    spect_data_list.append(spect_data_i)


In [ ]:
def add_power_law_fit(ax, k_vals, E_vals, k_min, k_max, color):
    """Fits E(k) = c * k^p, calculates R^2, and plots the fit."""
    mask = (k_vals >= k_min) & (k_vals <= k_max)
    k_fit, E_fit = k_vals[mask], E_vals[mask]
    
    if len(k_fit) < 2: return
    
    # Linear regression in log-log space
    log_k, log_E = np.log10(k_fit), np.log10(E_fit)
    slope, intercept, r_value, _, _ = linregress(log_k, log_E)
    r_squared = r_value**2
    
    # Plot the fit line
    fit_line = (10**intercept) * (k_fit**slope)
    label = f'Fit: slope={slope:.2f}, $R^2$={r_squared:.2f}'
    ax.plot(k_fit, fit_line, color=color, linestyle='--', linewidth=2, label=label)


fig, ax = plt.subplots(figsize=(10, 7), dpi=300)
file_nums = sorted(spect_data_list[0]['file_num'].unique())
norm = mcolors.Normalize(vmin=min(file_nums), vmax=max(file_nums))
cmap = cm.coolwarm

bin_width = 1
for num, group in spect_data_list[0].groupby('file_num'):
    # if num > 1000:
    #     continue
    raw_k, raw_E = group[0].values, group[1].values
    
    num_bins = len(raw_k) // bin_width
    k_binned = raw_k[:num_bins*bin_width].reshape(-1, bin_width).mean(axis=1)
    E_binned = raw_E[:num_bins*bin_width].reshape(-1, bin_width).mean(axis=1)
    
    ax.plot(k_binned, E_binned, color=cmap(norm(num)))
    
    # if num == target_time:
    fit_k, fit_E = k_binned, E_binned

ax.set_xscale('log')
ax.set_yscale('log')

# --- Add Reference Lines ---
ax.plot(fit_k, 5e-13 * fit_k, 'k--', label=r'Ref: $k^{1}$') 
# ax.plot(fit_k[:25], 1e-4 * np.power(fit_k[:25].astype(float), -4), 'm--', label=r'Ref: $k^{-4}$')

ax.axvline(x = 3, color = "purple", linestyle = "--", label="k=3")
# # 1024:
# # add_power_law_fit(ax, fit_k, fit_E, k_min=1.0, k_max= 60.0, color='gray')
# add_power_law_fit(ax, fit_k, fit_E, k_min=1.0, k_max=20.0, color='blue')
# # add_power_law_fit(ax, fit_k, fit_E, k_min=60.0, k_max=512.0, color='orange')
# # add_power_law_fit(ax, fit_k, fit_E, k_min=60.0, k_max=220.0, color='green')

# 512
# add_power_law_fit(ax, fit_k, fit_E, k_min=1.0, k_max= 30.0, color='gray')
# add_power_law_fit(ax, fit_k, fit_E, k_min=1.0, k_max=8.0, color='blue')
add_power_law_fit(ax, fit_k, fit_E, k_min=20.0, k_max=255.0, color='orange')
add_power_law_fit(ax, fit_k, fit_E, k_min=100.0, k_max=255.0, color='green')


# 2048:
# add_power_law_fit(ax, fit_k, fit_E, k_min=1.0, k_max= 50.0, color='gray')
# add_power_law_fit(ax, fit_k, fit_E, k_min=1.0, k_max=13.0, color='blue')
# add_power_law_fit(ax, fit_k, fit_E, k_min=100.0, k_max=1024.0, color='orange')
# add_power_law_fit(ax, fit_k, fit_E, k_min=100.0, k_max=400.0, color='green')


ax.set_xlabel('k')
ax.set_ylabel('E(k)')

# Legend for the fits and reference lines
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[-4:], labels[-4:], loc='lower left')

sm = cm.ScalarMappable(cmap=cmap, norm=norm)
fig.colorbar(sm, ax=ax, label='Simulation Step')

# plt.title(f"Incflo WENO5 Kolmogorov Energy Spectrum (Bin Width = {bin_width})\n(Nx=1024, visc = 1e-4, dt = 1.6e-4, cell_depth = 4e-5, ts = false)")
plt.title(f"Kolmogorov flow using Incflo WENO5 (Bin Width = {bin_width})\n(Nx=512, visc = 1e-4, dt = 3e-4, cell_depth = 1e-5, F_0 = .1, n = 3)")
# plt.title(f"Incflo WENO5 Kolmogorov Energy Spectrum (Bin Width = {bin_width})\n(Nx=2048, visc = 1e-4, dt = 8.1e-5, cell_depth = 1.6e-4, ts = false)")
plt.show()